# Step 2.3 — YOLOv5 Camera Detection (Upgraded) ✅

| | |
|---|---|
| **Input** | `output/step_1/camera_meta.json` (from Step 1.1) |
| **Outputs** | `output/step_2/yolo/<sample>/<camera>.json` — 2D bounding boxes |
| | `output/step_2/yolo_empty_detection_report.csv` — diagnostic breakdown |
| **Used by** | Step 2.3.1 (global 3D projection — see next notebook), Step 4.3 (camera tracking) |

---

### Fixed
- Path now points to `STEP1_DIR / "camera_meta.json"` directly (no emoji subfolder — matches the upgraded Step 1.1 output location).
- Added a diagnostic that breaks down your 382 empty-detection files **by camera channel**. If they cluster on back-facing cameras, that's mostly genuine empty scenes. If spread evenly, it points to the model's recall limit.

In [1]:
#pip install ultralytics

In [2]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 — Verify config.py exists
# ─────────────────────────────────────────────────────────────────

from pathlib import Path

if not Path("config.py").exists():
    raise FileNotFoundError("config.py not found. Copy it from the repo root.")

from config import DATA_ROOT, STEP1_DIR, STEP2_DIR

YOLO_OUT_DIR = STEP2_DIR / "yolo"
YOLO_OUT_DIR.mkdir(parents=True, exist_ok=True)

cam_meta_path = STEP1_DIR / "camera_meta.json"
if not cam_meta_path.exists():
    raise FileNotFoundError(f"camera_meta.json not found at {cam_meta_path} — run Step 1.1 first.")

print(f"✅ cam_meta_path : {cam_meta_path}")
print(f"✅ YOLO_OUT_DIR  : {YOLO_OUT_DIR}")

config.py loaded. PROJECT_ROOT = F:\Sensor fusion Research
DATA_ROOT   = F:\Sensor fusion Research\DATA SET\archive
OUTPUT_ROOT = F:\Sensor fusion Research\output
✅ cam_meta_path : F:\Sensor fusion Research\output\step_1\camera_meta.json
✅ YOLO_OUT_DIR  : F:\Sensor fusion Research\output\step_2\yolo


In [3]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — Run YOLOv5 on all camera images
# ─────────────────────────────────────────────────────────────────

import torch, json, cv2
from tqdm import tqdm

with open(cam_meta_path) as f:
    cam_meta_data = json.load(f)

# NOTE: to upgrade to YOLOv8 instead, replace this block with:
#   from ultralytics import YOLO
#   model = YOLO('yolov8n.pt')
# and change the inference call in the loop below to: results = model(img_rgb, conf=0.25)
# ------->model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.conf = 0.35
model.iou = 0.45
# FIXED: only report vehicle/person classes (COCO ids), not all 80 —
# was reporting benches, potted plants, traffic lights, etc.
# 0 person, 1 bicycle, 2 car, 3 motorcycle, 5 bus, 7 truck
model.classes = [0, 1, 2, 3, 5, 7]

total_detections = 0
empty_files = []

#for sample_id, cam_views in tqdm(cam_meta_data.items(), desc="Running YOLOv5"):
for sample_id, cam_views in tqdm(cam_meta_data.items(), desc="Running YOLOv8"):
    for cam_name, entry in cam_views.items():
        try:
            img_path = DATA_ROOT / entry["filename"]
            if not img_path.exists():
                print(f"❌ Image not found: {img_path}")
                continue

            img = cv2.imread(str(img_path))
            if img is None:
                print(f"❌ Failed to load: {img_path}")
                continue

            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            #results = model(img_rgb, size=640)

            results = model(img_rgb, imgsz=640, conf=model.conf, classes=model.classes)
            
            det_list = []
            
            for r in results:
                boxes = r.boxes
            
                for b in boxes:
                    det_list.append({
                        "xmin": int(b.xyxy[0][0].item()),
                        "ymin": int(b.xyxy[0][1].item()),
                        "xmax": int(b.xyxy[0][2].item()),
                        "ymax": int(b.xyxy[0][3].item()),
                        "confidence": float(b.conf[0].item()),
                        "class_id": int(b.cls[0].item()),
                        "class_name": model.names[int(b.cls[0].item())]
                    })
            
            #preds = results.pandas().xyxy[0]
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            results = model(img_rgb, imgsz=640, conf=model.conf, classes=model.classes)
            
            det_list = []
            
            for r in results:
                for b in r.boxes:
                    det_list.append({
                        "xmin": int(b.xyxy[0][0].item()),
                        "ymin": int(b.xyxy[0][1].item()),
                        "xmax": int(b.xyxy[0][2].item()),
                        "ymax": int(b.xyxy[0][3].item()),
                        "confidence": float(b.conf[0].item()),
                        "class_id": int(b.cls[0].item()),
                        "class_name": model.names[int(b.cls[0].item())]
                    })
            
            out_dir = YOLO_OUT_DIR / sample_id
            out_dir.mkdir(parents=True, exist_ok=True)
            
            out_file = out_dir / f"{cam_name}.json"
            
            with open(out_file, "w") as f:
                json.dump(det_list, f, indent=2)
            
            total_detections += len(det_list)
            
            if len(det_list) == 0:
                empty_files.append({
                    "sample_id": sample_id,
                    "camera": cam_name
                })

           # det_list = []
            #for _, row in preds.iterrows():
                #det_list.append({
                    #'xmin': int(row['xmin']), 'ymin': int(row['ymin']),
                    #'xmax': int(row['xmax']), 'ymax': int(row['ymax']),
                    #'confidence': float(row['confidence']),
                    #'class_id': int(row['class']),
                    #'class_name': row['name']
                #})

            out_dir = YOLO_OUT_DIR / sample_id
            out_dir.mkdir(parents=True, exist_ok=True)
            out_file = out_dir / f"{cam_name}.json"
            with open(out_file, 'w') as f:
                json.dump(det_list, f, indent=2)

            total_detections += len(det_list)
            if len(det_list) == 0:
                empty_files.append({"sample_id": sample_id, "camera": cam_name})

        except KeyError as e:
            print(f"❌ Missing key in metadata: {e}")
        except Exception as e:
            print(f"❌ Unexpected error for {entry}: {e}")

print(f"\n✅ All YOLO detections saved to: {YOLO_OUT_DIR}")
print(f"   Total detections: {total_detections}")
print(f"   Empty files: {len(empty_files)}")

Running YOLOv8:   0%|          | 0/404 [00:00<?, ?it/s]

0: 384x640 2 cars, 1 truck, 257.4ms


Speed: 34.6ms preprocess, 257.4ms inference, 43.3ms postprocess per image at shape (1, 3, 384, 640)


0: 384x640 2 cars, 1 truck, 68.9ms


Speed: 4.0ms preprocess, 68.9ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)


...

0: 384x640 (no detections), 49.2ms


Speed: 3.0ms preprocess, 49.2ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)


Running YOLOv8: 100%|██████████| 404/404 [06:21<00:00,  1.00it/s]

Running YOLOv8: 100%|██████████| 404/404 [06:21<00:00,  1.06it/s]


✅ All YOLO detections saved to: F:\Sensor fusion Research\output\step_2\yolo
   Total detections: 12090
   Empty files: 1354


## Diagnostic: WHY are detections empty? Breakdown by camera channel

In [4]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 — Break down empty detections by camera channel
# If BACK cameras dominate -> mostly genuine empty road scenes
# If spread evenly -> points to model recall limit (worth upgrading model size)
# ─────────────────────────────────────────────────────────────────

import pandas as pd

empty_df = pd.DataFrame(empty_files)

if len(empty_df) > 0:
    breakdown = empty_df["camera"].value_counts().reset_index()
    breakdown.columns = ["camera", "empty_count"]
    breakdown["pct_of_empties"] = (breakdown["empty_count"] / len(empty_df) * 100).round(1)

    print(f"Total empty detection files: {len(empty_df)}\n")
    display(breakdown)

    back_cams = {"CAM_BACK", "CAM_BACK_LEFT", "CAM_BACK_RIGHT"}
    back_pct = breakdown[breakdown["camera"].isin(back_cams)]["pct_of_empties"].sum()

    print(f"\n{'👉 ' if back_pct > 60 else '⚠️ '}{back_pct:.0f}% of empty detections are on back-facing cameras.")
    if back_pct > 60:
        print("   This is mostly consistent with genuinely empty rear scenes — not a bug.")
    else:
        print("   Empties are spread across front and back cameras — this points to the")
        print("   YOLOv5s model's recall limit rather than empty scenes. Consider upgrading")
        print("   to yolov5m/yolov5l, or YOLOv8, for better small/distant-object recall.")

    report_path = STEP2_DIR / "yolo_empty_detection_report.csv"
    breakdown.to_csv(report_path, index=False)
    print(f"\n📄 Saved: {report_path}")
else:
    print("✅ No empty detection files.")

Total empty detection files: 1354



,camera,empty_count,pct_of_empties
0,CAM_BACK_LEFT,430,31.8
1,CAM_BACK_RIGHT,334,24.7
2,CAM_FRONT_LEFT,328,24.2
3,CAM_FRONT_RIGHT,126,9.3
4,CAM_BACK,82,6.1
5,CAM_FRONT,54,4.0



👉 63% of empty detections are on back-facing cameras.
   This is mostly consistent with genuinely empty rear scenes — not a bug.

📄 Saved: F:\Sensor fusion Research\output\step_2\yolo_empty_detection_report.csv
